In [28]:
import sys
import os
import h5py
from pathlib import Path
from datetime import datetime

SRC = Path.cwd().parent
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from helpers.config import *
from model_training.model import *
from helpers.h5file_helpers import *

import torch
from torch.utils.data import Dataset, random_split, DataLoader
import torchvision.models as models
import torch.nn as nn
from sklearn.metrics import confusion_matrix

import warnings
warnings.filterwarnings("ignore")

## Glitches

In [23]:
with h5py.File(glitch_ehm_dataset_path, "r") as h5:
    qt_array = h5["QT"][:]

In [24]:
snrs = np.max(qt_array, axis=(2,3,4))
snrs = snrs[:, 2]
mask = snrs > 10

In [25]:
with h5py.File(glitch_ehm_dataset_path, "r") as h5:
    X_array = h5["X"][mask]       # shape (N, 512, 512)
    Y_array = h5["Y"][mask]       # shape (N, 512, 512)
    Z_array = h5["Z"][mask] 
    level_array = h5["level"][mask]
    beta_array = h5["beta"][mask]
    inj_point_array = h5["inj_point"][mask]
    qt_array = h5["QT"][mask]
    t_qscan_array = h5["t_qscan"][mask]
    f_qscan_array = h5["f_qscan"][mask]
    tcen_array = h5['tcen'][mask]

In [30]:
path = GLITCH_DATASETS / 'glitch_ehm_dataset2.h5'

if not os.path.exists(path):
    h5 = create_glitch_dataset(path, X_array.shape[1])

    tdi_dict = {}
    tdi_dict['X'] = X_array
    tdi_dict['Y'] = Y_array
    tdi_dict['Z'] = Z_array

    append_glitch_sample(h5, tdi_dict, level_array, beta_array, inj_point_array)
    h5.close()

    h5 = create_imageset(path, 5, 256, keys=['QT', 't_qscan', 'f_qscan'])
    append_image(h5, qt_array, t_qscan_array, f_qscan_array, tcen_array, keys=['QT', 't_qscan', 'f_qscan'])
    h5.close()


## Mixed

In [31]:
with h5py.File(mixed_ehm_dataset_path, "r") as h5:
    level_array = h5["level"][:]
    beta_array = h5["beta"][:]

mask = (level_array > 5e-10) & (beta_array > 5e-2)

In [35]:
with h5py.File(mixed_ehm_dataset_path, "r") as h5:
    X_array = h5["X"][mask]       # shape (N, 512, 512)
    Y_array = h5["Y"][mask]       # shape (N, 512, 512)
    Z_array = h5["Z"][mask] 
    m1_array = h5["m1"][mask]
    m2_array = h5["m2"][mask]
    d_array = h5["d"][mask]
    spin1_array = h5["spin1"][mask]
    spin2_array = h5["spin2"][mask]
    gw_beta_array = h5["gw_beta"][mask]
    gw_lambda_array = h5["gw_lambda"][mask]
    level_array = h5["level"][mask]
    beta_array = h5["beta"][mask]
    inj_point_array = h5["inj_point"][mask]
    sep_array = h5["sep"][mask]
    qt_array = h5["QT"][mask]
    t_qscan_array = h5["t_qscan"][mask]
    f_qscan_array = h5["f_qscan"][mask]
    tcen_array = h5['tcen'][mask]

In [36]:
path = MIXED_DATASETS / 'mixed_ehm_dataset2.h5'

if not os.path.exists(path):
    h5 = create_mixed_dataset(path, X_array.shape[1])

    tdi_dict = {}
    tdi_dict['X'] = X_array
    tdi_dict['Y'] = Y_array
    tdi_dict['Z'] = Z_array

    append_mixed_sample(h5, tdi_dict, m1_array, m2_array, d_array, spin1_array, spin2_array,
                        gw_beta_array, gw_lambda_array, level_array, beta_array, inj_point_array, sep_array)
    h5.close()

    h5 = create_imageset(path, 5, 256, keys=['QT', 't_qscan', 'f_qscan'])
    append_image(h5, qt_array, t_qscan_array, f_qscan_array, tcen_array, keys=['QT', 't_qscan', 'f_qscan'])
    h5.close()